# **Coeficiente de localización Baja California Sur**

El coeficiente de localización (LQ) es un indicador estadístico que permite evaluar qué tan concentrada está una industria, ocupación o grupo demográfico determinado en una zona geográfica concreta, la cual toma como referencia una región de mayor tamaño que suele ser nacional o regional.

Para el siguiente análisis el espacio demográfico determinado es el estado de Baja California Sur y como región de referencia de mayor tamaño el país.

El análisis busca responder a las siguientes preguntas: 

1. ¿En qué sectores destaca laboralmente el estado?
2. ¿La economía estatal se diversificó o concentro?
3. ¿Qué sectores podrían abastecer una demanda fuera del estado?
4. ¿El estado se hizo más especializado en algún sector?
5. ¿El estado se hizo menos especializado en algún sector?

# **Fuente**

- **Fuente:** INEGI, Censos Económicos 2024.

Ver `README.md` para el detalle completo de la consulta y notas metodológicas.

# **Código y resultados**

In [1]:
# Importar librería

import pandas as pd

# Cargar datos

datos_bcs = pd.read_csv('../data/datos_bcs.csv')

# Renombrar columnas

datos_bcs = datos_bcs.rename(columns={
    'Año Censal': 'año',
    'Entidad': 'entidad',
    'Actividad económica': 'sector',
    'H001A Personal ocupado total': 'pot'
})

In [2]:
# Función para calcular el coeficiente de localización por año

def coeficiente_localizacion(datos, año, entidad_estado, entidad_nacional):
    
    # 1. Filtrar por año
    
    df_año = datos.query('año == @año')
    
    # 2. Separar en dos dataframes nacional y estado
    
    nacional = df_año.query('entidad == @entidad_nacional')[['sector', 'pot']]
    estado = df_año.query('entidad == @entidad_estado')[['sector', 'pot']]
    
    # 3. Renombrar columnas de personal ocupado en ambos dataframe por pot_nacional y pot_bcs
    
    nacional = nacional.rename(columns={'pot': 'pot_nacional'})
    estado = estado.rename(columns={'pot': 'pot_bcs'})
    
    # 4. Combinar ambos dataframes mediante sector
    
    df_bcs = pd.merge(nacional, estado, on='sector', how='left')
    
    # 5. Calcular total_nacional y total_bcs (total del personal ocupado)
    
    total_nacional = df_bcs['pot_nacional'].sum()
    total_bcs = df_bcs['pot_bcs'].sum()
    
    # 6. Calcular LQ, empleo_esperado u empleo_excedente
    
    df_bcs['LQ'] = (df_bcs['pot_bcs'] / total_bcs) / (df_bcs['pot_nacional'] / total_nacional)
    df_bcs['empleo_esperado'] = (df_bcs['pot_nacional'] / total_nacional) * total_bcs
    df_bcs['empleo_excedente'] = df_bcs['pot_bcs'] - df_bcs['empleo_esperado']
    
    return df_bcs.set_index('sector')

In [3]:
# Ingresar datos a la función y obtener resultados

# 1. Crear lista de años a análizar y diccionario para guardar resultados

# 1.1. Lista de años

lista_años = [2008, 2013, 2018, 2023]

# 1.2. Diccionario de resultados

resultados = {}

# 2. Bucle for para ingresar datos a la función

for año in lista_años:
    resultados[año] = coeficiente_localizacion(datos_bcs, año, '03 Baja California Sur', '00 Total Nacional')

In [4]:
# Personal ocupado total por año de Baja California Sur y Nacional

# 1. Lista vacía para guardar los DataFrames

lista_pot = []

# 2. Bucle for para obtener los DataFrames por año y solo con las columnas pot_nacional y pot_bcs

for año in lista_años:
    tabla = resultados[año][['pot_nacional', 'pot_bcs']]
    lista_pot.append(tabla)
    
# 3. Concatenar los Dataframes en uno solo

pot = pd.concat(lista_pot, axis=1, keys=lista_años)

# 4. Mostrar tabla

pot

2008           \
                                                   pot_nacional  pot_bcs   
sector                                                                     
Sector 11 Agricultura, cría y explotación de an...     180083.0   9865.0   
Sector 21 Minería                                      142325.0   3935.0   
Sector 22 Generación, transmisión, distribución...     235688.0   2396.0   
Sector 23 Construcción                                 704640.0   8447.0   
Sector 31-33 Industrias manufactureras                4661062.0  11401.0   
Sector 43 Comercio al por mayor                       1107295.0   7532.0   
Sector 46 Comercio al por menor                       5027463.0  35542.0   
Sector 48-49 Transportes, correos y almacenamiento     718062.0   6649.0   
Sector 51 Información en medios masivos                293550.0   1571.0   
Sector 52 Servicios financieros y de seguros           480557.0    412.0   
Sector 53 Servicios inmobiliarios y de alquiler...     234548.0   2959.0   
Sector 54 Servicios profesionales, científicos ...     570637.0   3309.0   
Sector 55 Dirección y administración de grupos ...      26310.0      NaN   
Sector 56 Servicios de apoyo a los negocios y m...    1365509.0   9518.0   
Sector 61 Servicios educativos                         632422.0   2631.0   
Sector 62 Servicios de salud y de asistencia so...     584046.0   3805.0   
Sector 71 Servicios de esparcimiento culturales...     196793.0   1425.0   
Sector 72 Servicios de alojamiento temporal y d...    1743482.0  31985.0   
Sector 81 Otros servicios excepto actividades g...    1212362.0   8516.0   

                                                           2013           \
                                                   pot_nacional  pot_bcs   
sector                                                                     
Sector 11 Agricultura, cría y explotación de an...     188566.0   7066.0   
Sector 21 Minería                                      166548.0      NaN   
Sector 22 Generación, transmisión, distribución...     220929.0      NaN   
Sector 23 Construcción                                 569856.0   4886.0   
Sector 31-33 Industrias manufactureras                5073432.0  11067.0   
Sector 43 Comercio al por mayor                       1286393.0   9744.0   
Sector 46 Comercio al por menor                       5103255.0  37744.0   
Sector 48-49 Transportes, correos y almacenamiento     772333.0   5740.0   
Sector 51 Información en medios masivos                287617.0   1297.0   
Sector 52 Servicios financieros y de seguros           478776.0    690.0   
Sector 53 Servicios inmobiliarios y de alquiler...     251970.0   3024.0   
Sector 54 Servicios profesionales, científicos ...     606396.0   3638.0   
Sector 55 Dirección y administración de grupos ...      42758.0      NaN   
Sector 56 Servicios de apoyo a los negocios y m...    1714385.0  10159.0   
Sector 61 Servicios educativos                         759871.0   3830.0   
Sector 62 Servicios de salud y de asistencia so...     614147.0   4547.0   
Sector 71 Servicios de esparcimiento culturales...     231579.0   2675.0   
Sector 72 Servicios de alojamiento temporal y d...    1943437.0  35674.0   
Sector 81 Otros servicios excepto actividades g...    1264110.0   9816.0   

                                                           2018           \
                                                   pot_nacional  pot_bcs   
sector                                                                     
Sector 11 Agricultura, cría y explotación de an...     233554.0   7239.0   
Sector 21 Minería                                      190685.0   4061.0   
Sector 22 Generación, transmisión, distribución...     216300.0   1752.0   
Sector 23 Construcción                                 676301.0  11389.0   
Sector 31-33 Industrias manufactureras                6493020.0  12106.0   
Sector 43 Comercio al por mayor                       1582933.0  12057.0   
Sector 46 Comercio al por men

In [5]:
# Coeficiente de localización por sector y año

# 1. Crear un DataFrame vacío

lq = pd.DataFrame()

# 2. Bucle for para DataFrame del coeficiente de localización por año

for año in lista_años:
    lq[año] = resultados[año]['LQ']
    
# 3. Redondear LQ a tres decimales

lq = lq.round(3)

# 4. Mostrar tabla

lq

,2008,2013,2018,2023
sector,,,,
"Sector 11 Agricultura, cría y explotación de animales, aprovechamiento forestal, pesca y caza",7.255,5.333,3.737,4.005
Sector 21 Minería,3.662,NaN,2.568,NaN
"Sector 22 Generación, transmisión, distribución y comercialización de energía eléctrica, suministro de agua y de gas natural por ductos al consumidor final",1.346,NaN,0.977,NaN
Sector 23 Construcción,1.588,1.220,2.031,2.469
Sector 31-33 Industrias manufactureras,0.324,0.310,0.225,0.244
Sector 43 Comercio al por mayor,0.901,1.078,0.918,0.924
Sector 46 Comercio al por menor,0.936,1.053,0.916,0.987
"Sector 48-49 Transportes, correos y almacenamiento",1.226,1.058,0.959,0.814
Sector 51 Información en medios masivos,0.709,0.642,0.480,0.422
